# Azure OpenAI — Chat on Private Data (Simple RAG)
Load documents → Store vectors → Ask questions

## Step 1 · Install dependencies

In [ ]:
!pip install \
  "httpx==0.27.2" \
  "openai==1.51.0" \
  "langchain==0.2.16" \
  "langchain-core==0.2.38" \
  "langchain-openai==0.1.23" \
  "langchain-community==0.2.16" \
  "langchain-text-splitters==0.2.4" \
  "faiss-cpu==1.8.0" \
  "tiktoken==0.7.0" \
  "python-dotenv==1.0.1" \
  "numpy" \
  "pandas" \
  "typing_extensions" \
  --force-reinstall --quiet

# ⚠️ RESTART KERNEL after running this cell

## Step 2 · Connect to Azure OpenAI
Create a `.env` file in the same folder and add:
```
AZURE_OPENAI_API_KEY=your_key_here
AZURE_OPENAI_ENDPOINT=https://your-resource.openai.azure.com/
```

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

load_dotenv()

llm = AzureChatOpenAI(
    azure_deployment="gpt-4o",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-02-01",
    temperature=0,
)

embeddings = AzureOpenAIEmbeddings(
    azure_deployment="text-embedding-3-small",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-02-01",
)

print("✅ Connected")

## Step 3 · Load your documents

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import TokenTextSplitter

# Load all .txt files from your data folder
loader    = DirectoryLoader("../data/qna/", glob="*.txt", loader_cls=TextLoader)
documents = loader.load()

splitter = TokenTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks   = splitter.split_documents(documents)

print(f"✅ Loaded {len(documents)} documents → {len(chunks)} chunks")

## Step 4 · Save vectors to disk
> Run this **once**. After this your vectors are saved — no need to re-run on restart.

In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(chunks, embeddings)
db.save_local("vector_store")

print("✅ Vectors saved to disk at ./vector_store/")

## Step 5 · Load vectors from disk
> On every restart, **skip Steps 3 & 4** and just run this cell.

In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.load_local("vector_store", embeddings, allow_dangerous_deserialization=True)

print("✅ Vectors loaded from disk")

## Step 6 · See what documents are stored

In [ ]:
all_docs = list(db.docstore._dict.values())

summary = {}
for doc in all_docs:
    src = os.path.basename(doc.metadata.get("source", "unknown"))
    summary[src] = summary.get(src, 0) + 1

df = pd.DataFrame(summary.items(), columns=["File", "Chunks"])
df.index += 1
print(f"Total chunks in DB: {len(all_docs)}\n")
display(df)

# Also print exact source paths (needed when deleting/updating)
print("\nExact source paths stored in DB:")
for s in sorted(set(d.metadata.get("source", "") for d in all_docs)):
    print(" ", s)

## Step 7 · Build the QA chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.messages import HumanMessage, AIMessage

prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using only the context below. If unsure, say you don't know.\n\nContext: {context}"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

def build_chain(vector_db):
    """Build chain from a given db — call this again after any add/delete/update."""
    retriever = vector_db.as_retriever(search_kwargs={"k": 4})
    return (
        {
            "context":      RunnableLambda(lambda x: x["question"]) | retriever | format_docs,
            "question":     RunnableLambda(lambda x: x["question"]),
            "chat_history": RunnableLambda(lambda x: x.get("chat_history", [])),
        }
        | prompt
        | llm
    )

chain = build_chain(db)
print("✅ QA chain ready")

## Step 8 · Ask questions

In [ ]:
chat_history = []   # keeps conversation context across cells

def ask(question):
    global chat_history
    response = chain.invoke({
        "question":     question,
        "chat_history": chat_history,
    })
    answer = response.content
    chat_history.extend([
        HumanMessage(content=question),
        AIMessage(content=answer),
    ])
    print(f"\n❓ {question}")
    print(f"💬 {answer}")

In [ ]:
ask("Can you give me a brief of the AMAZING ANDAMAN package?")

In [ ]:
ask("What's the fee for Paris package fee")

In [ ]:
ask("Give me a brief of the 6 Nights / 7 Days Swiss package.")

In [ ]:
# Reset conversation history
chat_history = []
print("✅ Chat history cleared")

---
## Step 9 · Update your data

Use these cells whenever your documents change.

> ⚠️ **Important:** Always run **Step 6** first to see the exact source path stored in the DB before deleting or updating.

In [ ]:
# ── ADD a new document ──────────────────────────────────────
from langchain_community.document_loaders import TextLoader   # ✅ correct import

new_doc    = TextLoader("../data/qna/Kerala_Package.txt").load()
new_chunks = splitter.split_documents(new_doc)
db.add_documents(new_chunks)
db.save_local("vector_store")

# ✅ Rebuild chain so retriever uses the updated index
chain = build_chain(db)

print(f"✅ Added {len(new_chunks)} new chunks")

In [ ]:
ask("Give me a brief Kerala especially i wanted Munnar details")

In [ ]:
# ── DELETE a document ───────────────────────────────────────
# ⚠️ Run Step 6 first — copy the EXACT source path printed there

# Paste the exact path from Step 6 output here (do NOT use os.path.abspath)
file_to_remove = "../data/qna/Kerala_Package.txt"   # ← exact path from Step 6

all_docs  = list(db.docstore._dict.values())
kept_docs = [d for d in all_docs if d.metadata.get("source") != file_to_remove]

print(f"Before: {len(all_docs)} chunks")
print(f"After:  {len(kept_docs)} chunks")
print(f"Removed: {len(all_docs) - len(kept_docs)} chunks")

db = FAISS.from_documents(kept_docs, embeddings)
db.save_local("vector_store")

# ✅ Rebuild chain so retriever uses the updated index
chain = build_chain(db)

print(f"\n✅ Removed '{os.path.basename(file_to_remove)}' — {len(kept_docs)} chunks remaining")

In [ ]:
ask("Give me a brief Kerala especially i wanted Munnar details")

In [ ]:
# ── UPDATE (edited a file on disk, re-index it) ─────────────
# ⚠️ Run Step 6 first — copy the EXACT source path printed there
from langchain_community.document_loaders import TextLoader   # ✅ correct import

# Paste the exact path from Step 6 output here (do NOT use os.path.abspath)
file_to_update = "../data/qna/Paris.txt"   # ← exact path from Step 6

# 1) Remove old chunks
all_docs  = list(db.docstore._dict.values())
kept_docs = [d for d in all_docs if d.metadata.get("source") != file_to_update]
db        = FAISS.from_documents(kept_docs, embeddings)

# 2) Add updated file
updated_chunks = splitter.split_documents(TextLoader(file_to_update).load())
for chunk in updated_chunks:
    chunk.metadata["source"] = file_to_update   # preserve exact path
db.add_documents(updated_chunks)
db.save_local("vector_store")

# ✅ Rebuild chain so retriever uses the updated index
chain = build_chain(db)

print(f"✅ Updated '{os.path.basename(file_to_update)}'")

In [ ]:
ask("What's the fee for Paris package fee")

---
## Quick Reference

| What | When to run |
|---|---|
| Step 1 | **Once only** — install packages, then restart kernel |
| Steps 2 | **Every session** — imports and Azure connection |
| Steps 3–4 | **First time only** — load docs and save vectors to disk |
| Step 5 | **Every restart** — load saved vectors from disk |
| Step 6 | Any time — see what's stored + exact paths for delete/update |
| Step 7 | **Every session** — build QA chain |
| Step 8 | **Every session** — ask questions |
| Step 9 | When documents change — always rebuilds chain automatically |